## Introduzione
**PALINSESTO FUTURO + ENRICHMENT AUDITEL + ENRICHMENT LLM**

### OUTPUT:
1. `output_palinsesto_rai`: Palinsesto futuro arricchito per canali RAI
2. `output_palinsesto_competitor`: Palinsesto futuro arricchito per canali competitor

### FLOW DETTAGLIATO:
1. **Scraping palinsesto futuro (7 giorni)**  
   - Scarica i dati EPG per i prossimi 7 giorni da tivu.tv per tutti i canali target.
   - Parsing HTML e costruzione DataFrame con tutte le informazioni di programmazione.

2. **Normalizzazione programmi futuro**  
   - Uniforma i nomi dei programmi (rimozione varianti, normalizzazione titoli).
   - Mappatura manuale per eccezioni e correzioni.
   - Normalizzazione nomi canali secondo standard Auditel.

3. **Normalizzazione storico_programmi**  
   - Uniforma i dati storici Auditel per allineamento con i dati futuri.
   - Normalizzazione canali e date/orari.

4. **Aggregazione feature storiche**  
   - Calcolo delle metriche storiche (es. ascolti medi, share, durata media) per programma/canale/fascia oraria.
   - Aggregazione per età, giorno della settimana, fascia oraria.

5. **Join storico -> futuro**  
   - Unione tra palinsesto futuro e dati storici aggregati tramite chiavi normalizzate (programma, canale, fascia oraria).
   - Enrichment del palinsesto futuro con feature storiche.

6. **Enrichment LLM**  
   - Utilizzo di LLM (OpenAI) per arricchimento semantico: descrizione programmi, classificazione generi, estrazione keyword.
   - Batch processing e gestione timeout.

7. **Output finale**  
   - Scrittura dei dati arricchiti nelle tabelle di output per RAI e competitor.
   - Validazione e controllo qualità dei dati finali.

8. **Creazione vista del palinsesto unificato**
    - Aggregazione delle righe interrotte da programmi di intermezzo.

## Config

### Install

In [0]:
%run ./00_utility

In [0]:
%run ../FASE1/00_utils

In [0]:
# MAGIC %pip install beautifulsoup4 unidecode --quiet

In [0]:
import os
import re
import json
import time
import requests
import warnings
import pandas as pd

from bs4 import BeautifulSoup
from unidecode import unidecode
from datetime import date, timedelta, datetime
from delta.tables import DeltaTable

from pyspark.sql import functions as F
from pyspark.sql.functions import col, avg, regexp_replace

from openai import OpenAI

warnings.filterwarnings("ignore")

### Parameters

In [0]:
catalog = get_catalog()
print("Catalog: ", catalog)
# ============================================================
# TABLES
# ============================================================
AUDITEL_TABLE = f"{catalog}.whatif.storico_programmi"
TABLE_RAI = f"{catalog}.whatif.output_palinsesto_rai"
TABLE_COMP = f"{catalog}.whatif.output_palinsesto_competitor"
VW_PALINSESTO_FUTURO = f"{catalog}.whatif.vw_output_palinsesto_futuro"
# Parametri per le query nelle celle SQL (:param)
dbutils.widgets.text("TABLE_RAI", TABLE_RAI)
dbutils.widgets.text("TABLE_COMP", TABLE_COMP)
dbutils.widgets.text("VW_PALINSESTO_FUTURO", VW_PALINSESTO_FUTURO)

# ============================================================
# SCRAPING
# ============================================================
BASE_URL = "https://www.tivu.tv/epg_ajax_sat.aspx?d={day_offset}"
GIORNI_DA_SCARICARE = 7
SCARICO_SINGOLO_GIORNO = False # se True verrà scaricato il giorno oggi + GIORNI_DA_SCARICARE-1, altrimenti verranno scaricati tutti e 7 i giorni disponibili
DELAY_REQUEST = 0.5
DELAY_DETAIL = 0.2

# ============================================================
# CANALI
# ============================================================
CANALI_RAI = ["Rai 1", "Rai 2", "Rai 3"]
CANALI_COMPETITOR = ["Rete 4", "Canale 5", "Italia 1", "LA7", "TV8", "NOVE"]
CANALI_TIVU = CANALI_RAI + CANALI_COMPETITOR

MAP_CANALE_TIVU_AUDITEL = {
    "Rai 1": "Rai 1",
    "Rai 2": "Rai 2",
    "Rai 3": "Rai 3",
    "Rete 4": "Rete 4",
    "Canale 5": "Canale 5",
    "Italia 1": "Italia 1",
    "LA7": "La7",
    "TV8": "Tv8",
    "NOVE": "Nove"
}

CANALI_TARGET = [
    "Rai 1", "Rai 2", "Rai 3",
    "Rete 4", "Canale 5", "Italia 1",
    "La7", "Tv8", "Nove"
]

# ============================================================
# LLM
# ============================================================
AZURE_OPENAI_ENDPOINT = "https://llm-whatifp-coll.openai.azure.com/openai/v1/"
AZURE_OPENAI_KEY = dbutils.secrets.get(scope="whatif-palinsesti", key="azure-openai-key")
DEPLOYMENT_NAME = "gpt-5"  
BATCH_SIZE = 10
LLM_TIMEOUT = 60

### Mapping

In [0]:
ETA_COLS = {
    "15_24": "1st_Screen_LiveVOSDAL_Adulti_15_24",
    "25_34": "1st_Screen_LiveVOSDAL_Adulti_25_34",
    "35_44": "1st_Screen_LiveVOSDAL_Adulti_35_44",
    "45_54": "1st_Screen_LiveVOSDAL_Adulti_45_54",
    "55_64": "1st_Screen_LiveVOSDAL_Adulti_55_64",
    "65_69": "1st_Screen_LiveVOSDAL_Adulti_65_69",
    "70_74": "1st_Screen_LiveVOSDAL_Adulti_70_74",
    "75_plus": "1st_Screen_LiveVOSDAL_Adulti_75plus"
}

### Client OpenAI

In [0]:
client = OpenAI(
    api_key=AZURE_OPENAI_KEY,
    base_url=AZURE_OPENAI_ENDPOINT
)

## Scraping Palinsesto Futuro

In [0]:
all_rows = []
oggi = date.today()

# Determina la lista degli offset dei giorni da scaricare
if SCARICO_SINGOLO_GIORNO:
    offset_list = [GIORNI_DA_SCARICARE-1]
else:
    offset_list = range(GIORNI_DA_SCARICARE)

# Scarico e parse i dati per ciascun giorno
for offset in offset_list:
    giorno = oggi + timedelta(days=offset)
    try:
        html = fetch_epg(offset)  # Effettua la richiesta HTML
        rows = parse_epg(
            html,
            CANALI_TIVU,
            giorno
        )  # Estrae le righe dal palinsesto
        all_rows.extend(rows)
        print(f"{giorno}: {len(rows)} righe")

    except Exception as e:
        print(f"ERRORE {giorno}: {e}")
    time.sleep(DELAY_REQUEST)  # Breve delay tra le richieste

future_df = pd.DataFrame(all_rows)
# Elimina programmi della fascia notturna 00:00 - 06:59
future_df["ora"] = future_df["orario_inizio"].str[:2].astype(int)

#future_df = future_df[
#    ~future_df["ora"].between(0, 6)
#].copy()

print(f"Totale righe: {len(future_df):,}")

In [0]:
future_df.display()

## Normalizzazione Palinsesto Futuro

### Normalizzazione nomi programma

In [0]:
# Normalizza il nome del programma: prendi solo la parte prima del primo ' - '
future_df["Programma"] = (
    future_df["Programma"]
    .str.split(r"\s*-(?:\s+|$)", n=1, regex=True)
    .str[0]
    .str.strip()
)

# Lista dei programmi canonici da uniformare
canonical_programs = [
    "Angelus",
    "Amazing Stories",
    "Baywatch",
    "Blue Bloods",
    "Che ci faccio qui",
    "Crociere di nozze",
    "Codice:"
    "Di là dal fiume e tra gli alberi",
    "Elsbeth",
    "Festival circo Montecarlo",
    "Gli Omicidi del Lago",
    "Hot Ones Italia",
    "Hudson & Rex",
    "Il commissario Dupin",
    "Il commissario Lanz",
    "Il commissario Rex",
    "Il Fattore Umano",
    "Il Fattore Umano",
    "N.C.I.S",
    "N.C.I.S.",
    "Passato e Presente",
    "Protestantesimo",
    "Provaci ancora prof",
    "Racconti Criminali",
    "Ritorno a Las Sabinas",
    "Santa Messa",
    "Sorgente di vita",
    "Squadra Omicidi Barcellona",
    "S.W.A.T.",
    "The Beach",
    "Tour de France",
    "Un ciclone in convento",
    "Un giorno in Pretura"
]

conditions = []

for program in canonical_programs:

    if program.lower() == "festival circo montecarlo":
        pattern = r"\bfestival\s+circo(?:\s+di)?\s+montecarlo\b"
    else:
        pattern = re.escape(program)

    condizione = future_df["Programma"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    )

    conditions.append(condizione)

future_df["Programma"] = np.select(
    conditions,
    canonical_programs,
    default=future_df["Programma"]
)

# Normalizza il titolo del programma
future_df["programma_norm"] = future_df["Programma"].apply(normalize_title)

# Applica mapping manuale per eventuali eccezioni
future_df["programma_norm"] = [
    apply_manual_mapping(c, p)
    for c, p in zip(future_df["Canale"], future_df["programma_norm"])
]

# Rimuove eventuali ' -' residui dal nome del programma
future_df["Programma"] = future_df["Programma"].str.replace(" -", "", regex=False)

# Aggiunge uno spazio a TG2 20.30
future_df["Programma"] = future_df["Programma"].str.replace("TG220.30", "TG2 20.30 ", regex=False)

# Normalizza il nome del canale secondo la mappatura Auditel
future_df["canale_norm"] = (
    future_df["Canale"]
    .map(MAP_CANALE_TIVU_AUDITEL)
    .fillna(future_df["Canale"])
)

### Manipolazione orario/data

In [0]:

# Calcola la fascia oraria a partire dall'orario di inizio
future_df["fascia_oraria"] = (
    future_df["orario_inizio"]
    .apply(calcola_fascia)
)

# Calcola la durata in minuti tra orario di inizio e fine
future_df["durata_minuti"] = future_df.apply(
    lambda x: calcola_durata(
        x["orario_inizio"],
        x["orario_fine"]
    ),
    axis=1
)

# Estrae il giorno della settimana dalla data (0=lunedì, 6=domenica)
future_df["giorno_settimana"] = pd.to_datetime(
    future_df["Data"]
).dt.weekday

# Estrae l'ora (in formato intero) dall'orario di inizio
future_df["ora"] = (
    future_df["orario_inizio"]
    .str[:2]
    .astype(int)
)

In [0]:
future_df.display()

## Loading Palinsesto Storico

In [0]:
storico_df = read_df_programmi(
    table_name=AUDITEL_TABLE
)

## Normalizzazione Palinsesto Storico

In [0]:
# Filtra lo storico solo per i canali di interesse
storico_df = storico_df[storico_df["Canale"].isin(CANALI_TARGET)].copy()

In [0]:
# Normalizza il nome del canale secondo la mappatura Auditel
storico_df["canale_norm"] = (
    storico_df["Canale"]
)

# Estrae il giorno della settimana dalla data (0=lunedì, 6=domenica)
storico_df["giorno_settimana"] = pd.to_datetime(
    storico_df["Data"]
).dt.weekday

# Estrae l'ora di inizio (in formato intero, da secondi a ore)
storico_df["ora"] = (
    storico_df["ORA_INIZIO_TRX"] // 3600
).astype("Int64")

In [0]:
storico_df.display()

## Arricchimento da Storico

### Calcolo Feature Storiche

In [0]:
# ============================================================
# SHARE STORICO
# ============================================================
agg_share = (
    storico_df
    .groupby([
        "programma_norm",
        "canale_norm",
        "ora",
        "giorno_settimana"
    ])
    .agg(
        share_storico=("Share", "mean")
    )
    .reset_index()
)

# ============================================================
# GENERE
# ============================================================
agg_genere = (
    storico_df
    .groupby([
        "programma_norm",
        "canale_norm",
        "ora",
        "giorno_settimana"
    ])["DES_GENERE_ESTESA_INT"]
    .agg(lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else None)
    .reset_index(name="genere_predominante")
)

# ============================================================
# TARGET GENERE
# ============================================================
def compute_gender_target(group):
    uomini = group["1st_Screen_LiveVOSDAL_Uomini"].median()
    donne = group["1st_Screen_LiveVOSDAL_Donne"].median()

    if pd.isna(uomini) or pd.isna(donne):
        return None

    if abs(uomini - donne) < 5:
        return "Misto"

    return "Uomini" if uomini > donne else "Donne"

gender_rows = []
for keys, group in storico_df.groupby([
    "programma_norm",
    "canale_norm",
    "ora",
    "giorno_settimana"
]):
    gender_rows.append({
        "programma_norm": keys[0],
        "canale_norm": keys[1],
        "ora": keys[2],
        "giorno_settimana": keys[3],
        "target_genere": compute_gender_target(group)
    })

gender_df = pd.DataFrame(gender_rows)

# ============================================================
# TARGET ETA
# ============================================================
eta_rows = []
for keys, group in storico_df.groupby([
    "programma_norm",
    "canale_norm",
    "ora",
    "giorno_settimana"
]):
    medians = {}
    for label, colname in ETA_COLS.items():
        medians[label] = group[colname].median()
    eta_top = max(medians, key=medians.get)
    eta_rows.append({
        "programma_norm": keys[0],
        "canale_norm": keys[1],
        "ora": keys[2],
        "giorno_settimana": keys[3],
        "target_eta": eta_top
    })

eta_df = pd.DataFrame(eta_rows)

### Aggancio Feature Storiche

In [0]:
future_df = future_df.merge(
    agg_share,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

future_df = future_df.merge(
    agg_genere,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

future_df = future_df.merge(
    gender_df,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

future_df = future_df.merge(
    eta_df,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

In [0]:
%skip
future_df = future_df.merge(
    gender_df,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

future_df = future_df.merge(
    eta_df,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

## Divisione Rai e Competitor

In [0]:

df_rai_future = future_df[
    future_df["Canale"].isin(CANALI_RAI)
].copy()

df_comp_future = future_df[
    future_df["Canale"].isin(CANALI_COMPETITOR)
].copy()


print(f"RAI: {len(df_rai_future):,}")
print(f"COMP: {len(df_comp_future):,}")


In [0]:
if len(df_rai_future) == 0 or len(df_comp_future)==0:
    dbutils.notebook.exit("Dati mancanti per il palinsesto futuro")

In [0]:
df_rai_future = df_rai_future[[
    "Data",
    "canale_norm",
    "Programma",
    "programma_norm",
    "orario_inizio",
    "orario_fine",
    "fascia_oraria",
    "durata_minuti",
    "giorno_settimana",
    "ora",
    "share_storico",
    "genere_predominante",
    "target_genere",
    "target_eta"
]]

df_comp_future = df_comp_future[[
    "Data",
    "canale_norm",
    "Programma",
    "programma_norm",
    "orario_inizio",
    "orario_fine",
    "fascia_oraria",
    "durata_minuti",
    "giorno_settimana",
    "ora",
    "share_storico",
    "genere_predominante",
    "target_genere",
    "target_eta"
]]

In [0]:
df_rai_future = df_rai_future.rename(columns={
#     "programma_norm": "Programma",
     "canale_norm": "Canale"
 })

df_comp_future = df_comp_future.rename(columns={
#     "programma_norm": "Programma",
     "canale_norm": "Canale"
 })

In [0]:
df_rai_future.display()

In [0]:
df_comp_future.display()

## Arricchimento da LLM

In questa sezione, i dati dei palinsesti vengono arricchiti tramite un LLM che identifica automaticamente i programmi considerati "eventi forti" sulla base del contesto fornito.

In [0]:
# Carica il contesto Auditel per i canali RAI
ctx_rai = load_auditel_context(spark, ["Rai 1", "Rai 2", "Rai 3"])
# Carica il contesto Auditel per i canali competitor
ctx_comp = load_auditel_context(spark, CANALI_COMPETITOR)

# Crea una copia del dataframe futuro RAI e resetta l'indice
df_rai = df_rai_future.copy().reset_index(drop=True)
# Crea una copia del dataframe futuro competitor e resetta l'indice
df_comp = df_comp_future.copy().reset_index(drop=True)

print("RAI:", len(df_rai))
print("COMP:", len(df_comp))

In [0]:
import json
import pandas as pd

# Funzione per determinare se l'orario è notturno (< 6:00)
def is_night(val):
    try:
        hour = int(str(val).split(":")[0])
        return hour < 6
    except Exception:
        return False

# Funzione per chiamare il modello LLM e restituire la risposta
def call_llm(messages):
    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            response_format={"type": "json_object"} 
        )
        return response.choices[0].message.content
    except Exception as e:
        print("LLM ERROR:", e)
        return '{"items":[]}'

# Costruisce il prompt per il modello LLM
def build_prompt(ctx):
    return f"""
Sei un analista TV.

DATI AUDITEL DI RIFERIMENTO:
{ctx}

Restituisci SOLO JSON valido nel formato:

{{
  "items": [
    {{
      "gid": 0,
      "evento_forte": true
    }}
  ]
}}

Regole:
- usa solo i programmi forniti dall'utente
- non inventare dati
- non modificare i titoli
- se non sei sicuro, usa evento_forte=false
"""

# Parsing della risposta JSON del modello
def safe_json_load(raw):
    try:
        raw = raw.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        data = json.loads(raw)

        if isinstance(data, list):
            return data

        if isinstance(data, dict):
            return data.get("items", [])

        return []
    except Exception as e:
        print("JSON ERROR:", e)
        print("RAW:", raw[:1000])
        return []

# Arricchisce il dataframe con la risposta dell'LLM in batch
def enrich_with_llm(df, context, batch_size=BATCH_SIZE):
    system_prompt = build_prompt(context)
    programs = df.reset_index(drop=True).to_dict("records")
    results = {}
    n_batches = (len(programs) + batch_size - 1) // batch_size

    for b_start in range(0, len(programs), batch_size):
        batch = programs[b_start:b_start + batch_size]
        lines = []
        for i, p in enumerate(batch):
            gid = b_start + i
            lines.append(
                f"[gid={gid}] "
                f"Canale: {p.get('Canale', '')} | "
                f"Orario: {p.get('orario_inizio', '')}-{p.get('orario_fine', '')} | "
                f"Programma: {p.get('Programma', '')}"
            )

        user_prompt = "Analizza questi programmi TV:\n\n" + "\n".join(lines)
        print(f"Batch {b_start // batch_size + 1}/{n_batches}")

        raw = call_llm([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ])

        data = safe_json_load(raw)

        for item in data:
            gid = item.get("gid")
            if gid is None:
                continue
            try:
                gid = int(gid)
            except Exception:
                continue
            results[gid] = {
                "evento_forte": bool(item.get("evento_forte", False))
            }

    return results

# Crea una chiave univoca per ogni programma/canale
def make_key(r):
    return (
        str(r.get("Canale", "")).strip().lower(),
        str(r.get("Programma", "")).strip().lower()
    )

# Costruisce la mappatura tra chiave e risultato LLM
def build_enrichment_mapping(df, llm_results):
    mapping = {}
    for gid, row in enumerate(df.reset_index(drop=True).to_dict("records")):
        r = llm_results.get(gid, {})
        mapping[make_key(row)] = {
            "evento_forte": bool(r.get("evento_forte", False))
        }
    return mapping

# Applica la mappatura enrichment a una riga del dataframe
def apply_enrichment(row, mapping):
    feats = dict(mapping.get(make_key(row), {"evento_forte": False}))
    # Se il programma è notturno, non è evento forte
    if is_night(row.get("orario_inizio")):
        feats["evento_forte"] = False
    return pd.Series(feats)

# Funzione principale: arricchisce il dataframe, crea mapping e risultati LLM
def enrich_dataframe(df, ctx, label):
    llm_results = enrich_with_llm(df, ctx)
    mapping = build_enrichment_mapping(df, llm_results)
    enriched = df.copy()
    enriched[["evento_forte"]] = enriched.apply(
        lambda r: apply_enrichment(r, mapping),
        axis=1
    )
    print(f"\n{label}:")
    print(enriched["evento_forte"].value_counts(dropna=False))
    return enriched, mapping, llm_results

In [0]:
# RAI - LLM solo sui titoli unici, poi rimappo su tutte le righe
uni_rai = df_rai.drop_duplicates("Programma").reset_index(drop=True)
_, map_rai, _ = enrich_dataframe(uni_rai, ctx_rai, "RAI")
df_rai["evento_forte"] = df_rai.apply(lambda r: apply_enrichment(r, map_rai), axis=1)

In [0]:
df_rai.display()

In [0]:
# COMP - LLM solo sui titoli unici, poi rimappo su tutte le righe
uni_comp = df_comp.drop_duplicates("Programma").reset_index(drop=True)
_, map_comp, _ = enrich_dataframe(uni_comp, ctx_comp, "COMP")
df_comp["evento_forte"] = df_comp.apply(lambda r: apply_enrichment(r, map_comp), axis=1)

In [0]:
df_comp.display()

## Salvataggio Tabelle


In [0]:
df_rai = df_rai.drop_duplicates(['Data', 'Canale', 'orario_inizio', 'programma_norm'])
df_rai = spark.createDataFrame(df_rai)
df_rai = df_rai.filter(F.col('durata_minuti') < 500)
df_rai = df_rai.withColumn(
    'ID',
    F.concat(
        F.col('Canale'),
        F.lit('_'),
        F.col('Data'),
        F.lit('_'),
        F.col('programma_norm'),
        F.lit('_'),
        F.col('orario_inizio')
    )
)

# if table doesn't exist in UC --> create it
if not spark.catalog.tableExists(TABLE_RAI):
    df_rai.write.format("delta").saveAsTable(TABLE_RAI)
else:
    # otherwise do an upsert
    delta_table_rai = DeltaTable.forName(spark, TABLE_RAI)
    (
        delta_table_rai.alias("target")
        .merge(
            df_rai.alias("source"),
            "target.Data = source.Data AND target.Canale = source.Canale AND target.orario_inizio = source.orario_inizio"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
df_comp = df_comp.drop_duplicates(['Data', 'Canale', 'orario_inizio', 'programma_norm'])
df_comp = spark.createDataFrame(df_comp)
df_comp = df_comp.filter(F.col('durata_minuti') < 500)
df_comp = df_comp.withColumn(
    'ID',
    F.concat(
        F.col('Canale'),
        F.lit('_'),
        F.col('Data'),
        F.lit('_'),
        F.col('programma_norm'),
        F.lit('_'),
        F.col('orario_inizio')
    )
)

# if table doesn't exist in UC --> create it
if not spark.catalog.tableExists(TABLE_COMP):
    df_comp.write.format("delta").saveAsTable(TABLE_COMP)
else:
    # otherwise do an upsert
    delta_table_comp = DeltaTable.forName(spark, TABLE_COMP)
    (
        delta_table_comp.alias("target")
        .merge(
            df_comp.alias("source"),
            "target.Data = source.Data AND target.Canale = source.Canale AND target.orario_inizio = source.orario_inizio"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

## Creazione Vista

In [0]:
# Usiamo spark.sql perche' DDL non supporta variabili
spark.sql(f"""
CREATE VIEW IF NOT EXISTS {VW_PALINSESTO_FUTURO} AS 
-- CREATE OR REPLACE VIEW {VW_PALINSESTO_FUTURO} AS 
-- Palinsesto unificato: unisce in una sola riga sia i programmi con righe consecutive (split editoriali, gap ≈0) sia quelli interrotti da programmi più corti (spot, meteo...).
-- Palinsesto completo
WITH palinstesto AS (
    SELECT
        Canale, Data, Programma, programma_norm,
        orario_inizio, orario_fine, share_storico, evento_forte
    FROM {TABLE_RAI}
    UNION ALL
    SELECT
        Canale, Data, Programma, programma_norm,
        orario_inizio, orario_fine, share_storico, evento_forte
    FROM {TABLE_COMP} 
),
-- Calcolo dell'orario in secondi
base AS (
    SELECT
        Canale, Data, Programma, programma_norm, 
        orario_inizio, orario_fine, share_storico,
        (CAST(SPLIT(orario_inizio, ':')[0] AS INT) * 3600 + CAST(SPLIT(orario_inizio, ':')[1] AS INT) * 60) AS inizio_sec,
        (CAST(SPLIT(orario_fine, ':')[0] AS INT) * 3600 + CAST(SPLIT(orario_fine, ':')[1] AS INT) * 60) AS fine_sec_raw,
        evento_forte
    FROM palinstesto
),
-- Orario finale corretto per i programmi che iniziano prima di mezzanotte ma finiscono dopo
midnight_adjusted AS (
    SELECT
        *,
        CASE
            WHEN fine_sec_raw < inizio_sec
            THEN fine_sec_raw + 86400
            ELSE fine_sec_raw
        END AS fine_sec
    FROM base
),
-- Flag per i programmi consecutivi da aggregare e i programmi di interruzione
flagged AS (
    SELECT
        Canale, Data, Programma, programma_norm, 
        orario_inizio, 
        orario_fine,
        inizio_sec,
        fine_sec,
        share_storico,
        evento_forte,
        -- Flag programmi consecutivi da aggregare
        CASE
            WHEN inizio_sec - LAG(fine_sec) OVER w1 <= 900 -- se il gap tra due consecutivi e' <=15 min, viene considerato lo stesso programma
            THEN 0 ELSE 1
        END AS is_new_group,
        -- Flag programma interrutore
        CASE
            WHEN LAG(programma_norm) OVER w2 != programma_norm -- diverso dal precedente
            AND LAG(programma_norm) OVER w2 = LEAD(programma_norm) OVER w2 -- prima e dopo c'e' lo stesso programma
            AND inizio_sec - LAG(fine_sec) OVER w2 <= 300 -- inizia entro i 5 minuti dal precedente
            AND (LEAD(inizio_sec) OVER w2) - fine_sec <= 300 -- finisce entro i 5 minuti dal successivo
            AND (fine_sec - inizio_sec) <= 900 -- la riga stessa dura massimo 15 minuti
            THEN 1 
            ELSE 0
        END AS is_interruptor
    FROM midnight_adjusted
    WINDOW 
        w1 AS (PARTITION BY Data, Canale, programma_norm ORDER BY inizio_sec),
        w2 AS (PARTITION BY Data, Canale ORDER BY inizio_sec)
),
-- Flag per separare i gruppi di programmi
grouped AS (
    SELECT
        Canale, Data, Programma, programma_norm,
        orario_inizio, orario_fine,
        inizio_sec, fine_sec, share_storico,
        evento_forte,
        SUM(is_new_group) OVER (
            PARTITION BY Data, Canale, programma_norm
            ORDER BY inizio_sec
        ) AS grp
    FROM flagged
)
-- Raggruppamento e creazione ID
SELECT
    CONCAT(Canale, '_', CAST(Data AS STRING), '_', programma_norm, '_', MIN_BY(orario_inizio, inizio_sec)) AS ID,
    Canale, Data,
    MIN_BY(Programma, inizio_sec) AS Programma,
    MIN_BY(orario_inizio, inizio_sec) AS orario_inizio,
    MAX_BY(orario_fine, fine_sec) AS orario_fine,
    AVG(share_storico) AS share_storico,
    MAX(evento_forte) AS evento_forte
FROM grouped
GROUP BY Data, Canale, programma_norm, grp
ORDER BY orario_inizio;
"""
)

In [0]:
%sql
SELECT * FROM IDENTIFIER(:VW_PALINSESTO_FUTURO) 
ORDER BY Data DESC